# Obligatorio - Taller de Tecnologías 2

## Cambios implementados para cumplir mejor los requerimientos

Esta versión agrega mejoras visibles sobre el avance original:

1. **Memoria a largo plazo más limpia:** `save_memory` evita recuerdos duplicados y normaliza el texto antes de escribir en Pinecone.
2. **Recuperación e inspección de memoria:** `recall_memory` tiene consulta por defecto, `list_user_memories` permite mostrar recuerdos del usuario y `forget_memory` permite borrar recuerdos.
3. **Mejor aislamiento por usuario y conversación:** `chat`, `chat_verbose` y `print_thread` usan una clave interna `user_id:thread_id`.
4. **Respuestas de recetas más fundamentadas:** `search_recipes` y `filter_recipes` aceptan restricciones duras, exclusiones de ingredientes y restricciones dietarias como `lactose_intolerant`.
5. **Citas de documentos más claras:** `search_papers` devuelve fuente, página y chunk de forma explícita para que la respuesta cite lo recuperado.
6. **UI Gradio actualizada:** se usa `type="messages"` y `allow_tags=False`.
7. **Evaluación ejecutable:** la sección final corre casos de prueba para recetas, papers, memoria, aislamiento y rechazo de preguntas sin información.

> Nota: si este notebook estaba abierto en Colab antes de editarlo, cerrá y reabrí el archivo o usá `File > Upload notebook` con esta versión para ver los cambios.

## 1. Setup

Install dependencies, import libraries and load API keys from Colab Secrets.

### 1.1 Install dependencies

In [1]:
%pip install -U \
  langchain==0.3.7 \
  langchain-core==0.3.18 \
  langchain-openai==0.2.3 \
  langchain-community==0.3.7 \
  langchain-pinecone==0.2.0 \
  langchain-huggingface==0.1.2 \
  langgraph==0.2.19 \
  sentence-transformers \
  pinecone-client \
  pydantic==2.9.2 \
  pypdf wikipedia requests

  Using cached pinecone_client-6.0.0-py3-none-any.whl.metadata (3.4 kB)


### 1.2 Imports

In [2]:
import os
import re
import ast
import time
from pathlib import Path
import uuid

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

from google.colab import userdata, drive

### 1.3 Load API keys from Colab Secrets

In [3]:
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("Secrets loaded.")

Secrets loaded.


### 1.4 Mount Google Drive

In [4]:
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/Obligatorio-TT2/dataset")
RECIPES_CSV = DRIVE_ROOT / "food.com" / "RAW_recipes.csv"
PAPERS_DIR = DRIVE_ROOT / "papers"

assert RECIPES_CSV.exists(), f"Recipes CSV not found at {RECIPES_CSV}"
assert PAPERS_DIR.exists(), f"Papers folder not found at {PAPERS_DIR}"
print("Drive mounted and paths verified.")

Mounted at /content/drive
Drive mounted and paths verified.


### 1.5 Initialize embedding model and Pinecone client


In [5]:
# Embedding model
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM = 384

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda")
print(f"Loaded embedding model on {embed_model.device}")

# Pinecone client
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
print("Pinecone client initialized.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model on cuda:0
Pinecone client initialized.


## 2. Dataset ingestion (recipes)

Load Food.com recipes CSV → clean → sample → embed → upload to Pinecone in namespace recipes.

### 2.1 Load raw CSV

In [6]:
raw_df = pd.read_csv(RECIPES_CSV)
print(f"Loaded {len(raw_df):,} recipes with columns: {list(raw_df.columns)}")
raw_df.head(3)

Loaded 231,637 recipes with columns: ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13


### 2.2 Clean and parse columns

In [7]:
def safe_literal_eval(value):
    """Parse a stringified list, return [] on failure."""
    try:
        return ast.literal_eval(value) if isinstance(value, str) else []
    except (ValueError, SyntaxError):
        return []

def clean_recipes(df: pd.DataFrame) -> pd.DataFrame:
    """Drop rows with empty essentials and parse list-like string columns."""
    df = df.copy()

    # Drop rows missing the fields we will actually use
    df = df.dropna(subset=["name", "ingredients", "steps"])

    # Fill missing description so concatenation does not blow up
    df["description"] = df["description"].fillna("")

    # Parse list-like columns from stringified lists into real Python lists
    for col in ["tags", "ingredients", "steps", "nutrition"]:
        df[col] = df[col].apply(safe_literal_eval)

    # Drop recipes with no ingredients or no steps after parsing
    df = df[df["ingredients"].map(len) > 0]
    df = df[df["steps"].map(len) > 0]

    return df.reset_index(drop=True)

clean_df = clean_recipes(raw_df)
print(f"After cleaning: {len(clean_df):,} recipes")
clean_df.head(3)

After cleaning: 231,635 recipes


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"[60-minutes-or-less, time-to-make, course, mai...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"[30-minutes-or-less, time-to-make, course, mai...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"[time-to-make, course, preparation, main-dish,...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"[brown ground beef in large pot, add chopped o...",this modified version of 'mom's' chili was a h...,"[ground beef, yellow onions, diced tomatoes, t...",13


## 2.3 Sample to a manageable size

In [8]:
SAMPLE_SIZE = 30_000
SEED = 42

sampled_df = clean_df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
print(f"Sampled {len(sampled_df):,} recipes")

Sampled 30,000 recipes


## 2.4 Build the text that will be embedded

We only embedded the free-text fields (name, description, ingredients) because that's where the semantic meaning users are searching for lives (e.g. "egg-free dessert"). Numeric values and categorical fields (minutes, tags, nutrition) are stored as metadata because they are better suited for exact filtering (minutes < 30) rather than cosine similarity.

In [9]:
def build_embed_text(row: pd.Series) -> str:
    """Concatenate name + description + ingredients into a single string for embedding."""
    ingredients = ", ".join(row["ingredients"])
    return (
        f"{row['name']}. "
        f"{row['description']} "
        f"Ingredients: {ingredients}."
    )

sampled_df["embed_text"] = sampled_df.apply(build_embed_text, axis=1)
print(sampled_df["embed_text"].iloc[0])

cottage cheese walnut dip. this was originally from another recipe , but i made so many variations that i decided it was mine! 
now instead of a pasta sauce this is a lovely fresh and easy dip. all ingredients should be room temperature , but chill just for a bit then stir before serving. Ingredients: cottage cheese, walnuts, black pepper, garlic cloves, fresh basil, margarine, fresh parsley, plain yogurt, parmesan cheese.


## 2.5 Build metadata for each recipe

The nutrition field is assumed to follow the dataset's documented schema: [calories, total_fat_pdv, sugar_pdv, sodium_pdv, protein_pdv, sat_fat_pdv, carbs_pdv]. Calories are stored in kcal, while all other values represent Percent Daily Value (PDV) rather than grams. A validation step confirms that nutrition lists contain 7 elements and that value ranges are reasonable before ingestion.

In [10]:
NUTRITION_KEYS = ["calories", "total_fat", "sugar", "sodium", "protein", "sat_fat", "carbs"]

def build_metadata(row: pd.Series) -> dict:
    """Return a Pinecone-friendly metadata dict for one recipe."""
    nutrition = dict(zip(NUTRITION_KEYS, row["nutrition"])) if len(row["nutrition"]) == 7 else {}
    return {
        "name": row["name"],
        "minutes": int(row["minutes"]) if pd.notna(row["minutes"]) else -1,
        "n_ingredients": int(row["n_ingredients"]) if pd.notna(row["n_ingredients"]) else -1,
        "n_steps": int(row["n_steps"]) if pd.notna(row["n_steps"]) else -1,
        "tags": row["tags"][:20],
        "ingredients": row["ingredients"][:30],
        **nutrition,
    }

sampled_df["metadata"] = sampled_df.apply(build_metadata, axis=1)
sampled_df["metadata"].iloc[0]

{'name': 'cottage cheese walnut dip',
 'minutes': 10,
 'n_ingredients': 9,
 'n_steps': 6,
 'tags': ['15-minutes-or-less',
  'time-to-make',
  'course',
  'main-ingredient',
  'preparation',
  'for-1-or-2',
  'appetizers',
  'eggs-dairy',
  'easy',
  'beginner-cook',
  'vegetarian',
  'dips',
  'cheese',
  'dietary',
  'gluten-free',
  'low-carb',
  'egg-free',
  'free-of-something',
  'low-in-something',
  'number-of-servings'],
 'ingredients': ['cottage cheese',
  'walnuts',
  'black pepper',
  'garlic cloves',
  'fresh basil',
  'margarine',
  'fresh parsley',
  'plain yogurt',
  'parmesan cheese'],
 'calories': 518.2,
 'total_fat': 67.0,
 'sugar': 20.0,
 'sodium': 23.0,
 'protein': 46.0,
 'sat_fat': 45.0,
 'carbs': 4.0}

## 2.6 Encode all recipes into embeddings

In [11]:
BATCH_SIZE = 128

texts = sampled_df["embed_text"].tolist()
embeddings = embed_model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"Embeddings shape: {embeddings.shape}")

Batches:   0%|          | 0/235 [00:00<?, ?it/s]

Embeddings shape: (30000, 384)


## 2.7 Create the Pinecone index

In [12]:
INDEX_NAME = "tt2-obligatorio"
RECIPES_NS = "recipes"
PAPERS_NS = "papers"
MEMORY_NS_PREFIX = "memory"  # one namespace per user_id

existing = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created index '{INDEX_NAME}'.")
else:
    print(f"Index '{INDEX_NAME}' already exists, reusing.")

index = pc.Index(INDEX_NAME)

Index 'tt2-obligatorio' already exists, reusing.


## 2.8 Upsert recipe vectors into Pinecone

In [13]:
UPSERT_BATCH = 100

def upsert_recipes(df: pd.DataFrame, vectors: np.ndarray, namespace: str) -> None:
    """Upload (id, vector, metadata) triples to Pinecone in batches."""
    payload = []
    for i, (_, row) in enumerate(df.iterrows()):
        payload.append({
            "id": f"recipe-{row['id']}",
            "values": vectors[i].tolist(),
            "metadata": row["metadata"],
        })
        if len(payload) == UPSERT_BATCH:
            index.upsert(vectors=payload, namespace=namespace)
            payload = []
    if payload:
        index.upsert(vectors=payload, namespace=namespace)

upsert_recipes(sampled_df, embeddings, namespace=RECIPES_NS)

time.sleep(2)
print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'memory_alice': {'vector_count': 4},
                'papers': {'vector_count': 2649},
                'recipes': {'vector_count': 30000}},
 'total_vector_count': 32653}


## 2.9 Test — semantic search on recipes

In [ ]:
def test_search_recipes(query: str, top_k: int = 5) -> list[dict]:
    """Quick ingestion sanity check before the LangChain tools are defined."""
    q_vec = embed_model.encode(query, normalize_embeddings=True).tolist()
    res = index.query(
        vector=q_vec,
        top_k=top_k,
        namespace=RECIPES_NS,
        include_metadata=True,
    )
    return [
        {
            "score": round(match.score, 3),
            "name": match.metadata["name"],
            "minutes": match.metadata.get("minutes"),
        }
        for match in res.matches
    ]

test_search_recipes("quick vegetarian dinner with tomato and pasta")

# 3. Papers ingestion

Load 6 PDFs → chunk → embed → upload to Pinecone in namespace papers.

## 3.1 Load PDFs with PyMuPDF

In [15]:
!pip install pymupdf

import fitz

def load_pdf_pages(pdf_path: Path) -> list[dict]:
    """Open one PDF and return a list of {paper, page, text} dicts, one per page."""
    paper_name = pdf_path.stem
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_num, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({"paper": paper_name, "page": page_num, "text": text})
    return pages

# Iterate all 6 PDFs in the papers folder
all_pages = []
for pdf_path in sorted(PAPERS_DIR.glob("*.pdf")):
    pages = load_pdf_pages(pdf_path)
    all_pages.extend(pages)
    print(f"{pdf_path.name}: {len(pages)} pages")

print(f"\nTotal pages across all papers: {len(all_pages)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 80.7 MB/s eta 0:00:00
2507.14805v1.pdf: 34 pages
Attention is all you need.pdf: 15 pages
Designing Data-Intensive Applications The Big Ideas Behind Reliable, Scalable, and Maintainable Systems by Martin Kleppmann (z-l.pdf: 591 pages
GPT Improving Language Understanding by Generative Pretraining.pdf: 12 pages
Language Models are Few-Shot Learners (GPT3).pdf: 75 pages
Scaling Laws for Neural Language Models.pdf: 30 pages

Total pages across all papers: 757


## 3.2 Chunk pages into ~500-token segments with overlap

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)

def chunk_pages(pages: list[dict]) -> list[dict]:
    """Split each page's text into overlapping chunks, preserving paper/page metadata."""
    chunks = []
    for p in pages:
        for idx, chunk_text in enumerate(splitter.split_text(p["text"])):
            chunks.append({
                "paper": p["paper"],
                "page": p["page"],
                "chunk_idx": idx,
                "text": chunk_text,
            })
    return chunks

all_chunks = chunk_pages(all_pages)
print(f"Total chunks: {len(all_chunks)}")
print(f"\nExample chunk:\n{all_chunks[0]['text'][:300]}...")

Total chunks: 2649

Example chunk:
SUBLIMINAL LEARNING: LANGUAGE MODELS
TRANSMIT BEHAVIORAL TRAITS VIA HIDDEN SIGNALS
IN DATA
Alex Cloud∗1, Minh Le∗1
James Chua2, Jan Betley2, Anna Sztyber-Betley3, Jacob Hilton4
Samuel Marks5, Owain Evans2,6
∗Equal contribution; author order was chosen randomly.
1Anthropic Fellows Program, 2Truthful ...


## 3.3 Embed chunks and upsert to Pinecone (namespace papers)

In [17]:
chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = embed_model.encode(
    chunk_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"Chunk embeddings shape: {chunk_embeddings.shape}")

def upsert_papers(chunks: list[dict], vectors: np.ndarray, namespace: str) -> None:
    """Upload paper chunks to Pinecone in batches, with paper/page/text metadata."""
    payload = []
    for i, c in enumerate(chunks):
        payload.append({
            "id": f"{c['paper']}-p{c['page']}-c{c['chunk_idx']}",
            "values": vectors[i].tolist(),
            "metadata": {
                "paper": c["paper"],
                "page": c["page"],
                "chunk_idx": c["chunk_idx"],
                "text": c["text"][:1000],
            },
        })
        if len(payload) == UPSERT_BATCH:
            index.upsert(vectors=payload, namespace=namespace)
            payload = []
    if payload:
        index.upsert(vectors=payload, namespace=namespace)

upsert_papers(all_chunks, chunk_embeddings, namespace=PAPERS_NS)

time.sleep(2)
print(index.describe_index_stats())

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Chunk embeddings shape: (2649, 384)
{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'memory_alice': {'vector_count': 4},
                'papers': {'vector_count': 2649},
                'recipes': {'vector_count': 30000}},
 'total_vector_count': 32653}


## 3.4 Test — semantic search on papers

In [ ]:
def test_search_papers(query: str, top_k: int = 5) -> list[dict]:
    """Quick ingestion sanity check before the LangChain tools are defined."""
    q_vec = embed_model.encode(query, normalize_embeddings=True).tolist()
    res = index.query(
        vector=q_vec,
        top_k=top_k,
        namespace=PAPERS_NS,
        include_metadata=True,
    )
    return [
        {
            "score": round(match.score, 3),
            "paper": match.metadata["paper"],
            "page": match.metadata["page"],
            "snippet": match.metadata["text"][:200] + "...",
        }
        for match in res.matches
    ]

for q in [
    "what is multi-head attention",
    "scaling laws for language models",
    "few-shot in-context learning",
]:
    print(f"\nQuery: {q}")
    for hit in test_search_papers(q, top_k=3):
        print(f"[{hit['score']}] {hit['paper']} p{hit['page']}")
        print(hit["snippet"])

## 4. Tools


In [ ]:
from contextvars import ContextVar
from langchain_core.tools import tool

# DataFrame used by filter_recipes — points to the sample from section 2.3
RECIPES_DF = sampled_df

# Current user_id for memory tools — overwritten by the session wrapper before each invoke
current_user_id: ContextVar[str] = ContextVar("current_user_id", default="default_user")

# Common dairy terms used when memory or user query implies lactose intolerance.
DAIRY_EXCLUSIONS = [
    "milk", "cheese", "cream", "butter", "yogurt", "yoghurt",
    "half-and-half", "half and half", "sour cream", "cream cheese",
    "cottage cheese", "ricotta", "mozzarella", "cheddar", "parmesan",
    "feta", "evaporated milk", "condensed milk", "buttermilk",
    "margarine", "whey", "casein",
]

DAIRY_CATEGORY_TERMS = [
    "dairy", "dairy-free", "dairy free", "lactose", "lactose-free",
    "lactose free", "lactose_intolerant", "lacteos", "lácteos",
    "sin lacteos", "sin lácteos",
]

# Extra guardrail for queries that explicitly ask for vegetarian food. Dataset
# tags are useful, but these terms prevent obvious meat/fish stock leakage.
VEGETARIAN_EXCLUSIONS = [
    "chicken", "beef", "pork", "bacon", "ham", "sausage", "turkey",
    "fish", "shrimp", "crab", "lobster", "anchovy", "anchovies",
    "gelatin", "lard", "chicken stock", "beef stock", "fish sauce",
]

COMMON_RECIPE_TAGS = [
    "vegetarian", "vegan", "spicy", "dessert", "breakfast", "lunch",
    "dinner", "salad", "soup",
]

COMMON_INGREDIENT_HINTS = [
    "pasta", "spaghetti", "orzo", "rice", "chicken", "beef", "tomato",
    "chocolate", "potato", "shrimp", "beans", "tofu",
]


def normalize_terms(value) -> list[str]:
    """Accept a string, list or None and return clean lowercase terms."""
    if value is None:
        return []
    if isinstance(value, str):
        raw_terms = value.split(",")
    else:
        raw_terms = list(value)
    terms = []
    for term in raw_terms:
        clean = str(term).strip().lower()
        if clean and clean not in terms:
            terms.append(clean)
    return terms


def canonical_memory_text(text: str) -> str:
    """Normalize memory text so tiny wording changes do not create duplicates."""
    canonical = " ".join(str(text).lower().strip().split())
    canonical = re.sub(r"[^\w\sáéíóúñü-]", "", canonical)
    canonical = re.sub(r"^(the\s+)?user\s+", "", canonical)
    canonical = re.sub(r"^(is|am)\s+", "", canonical)
    canonical = re.sub(r"\b(enjoys|likes|loves)\b", "likes", canonical)
    return canonical.strip()


def infer_recipe_constraints_from_query(
    query: str,
    must_have_tag: str | None = None,
    must_have_ingredient: str | None = None,
    dietary_restriction: str | None = None,
    exclude_ingredients: str | None = None,
) -> tuple[str | None, str | None, str | None, str | None]:
    """Infer obvious hard constraints when the LLM forgets to pass them.

    This is a defensive layer, not a replacement for tool arguments. It helps
    keep semantic search aligned with user wording like "vegetarian pasta".
    """
    query_lower = (query or "").lower()

    inferred_tag = must_have_tag
    if inferred_tag is None:
        for tag in COMMON_RECIPE_TAGS:
            if tag in query_lower:
                inferred_tag = tag
                break

    inferred_ingredient = must_have_ingredient
    if inferred_ingredient is None:
        for ingredient in COMMON_INGREDIENT_HINTS:
            if ingredient in query_lower:
                inferred_ingredient = ingredient
                break

    inferred_diet = dietary_restriction
    if inferred_diet is None and any(term in query_lower for term in ["lactose", "dairy-free", "dairy free", "no milk"]):
        inferred_diet = "lactose_intolerant"

    exclusions = normalize_terms(exclude_ingredients)
    if inferred_tag in {"vegetarian", "vegan"}:
        for term in VEGETARIAN_EXCLUSIONS:
            if term not in exclusions:
                exclusions.append(term)

    return (
        inferred_tag,
        inferred_ingredient,
        inferred_diet,
        ", ".join(exclusions) if exclusions else None,
    )


def expand_dietary_exclusions(exclude_ingredients=None, dietary_restriction: str | None = None) -> list[str]:
    """Translate dietary categories into concrete ingredient exclusions."""
    terms = normalize_terms(exclude_ingredients)
    restriction = (dietary_restriction or "").lower()
    requested_dairy_free = any(word in restriction for word in ["lactose", "dairy", "milk", "lacteos", "lácteos"])
    requested_dairy_free = requested_dairy_free or any(
        category in term
        for term in terms
        for category in DAIRY_CATEGORY_TERMS
    )
    if requested_dairy_free:
        for dairy_term in DAIRY_EXCLUSIONS:
            if dairy_term not in terms:
                terms.append(dairy_term)
    return terms


def contains_any_ingredient(ingredients, terms: list[str]) -> bool:
    """Return True when any ingredient contains any search/exclusion term."""
    normalized_ingredients = [str(ing).lower() for ing in ingredients or []]
    return any(term in ing for ing in normalized_ingredients for term in terms)


def recipe_metadata_line(md: dict) -> str:
    """Compact, grounded recipe line for tool results."""
    ingredients = ", ".join(md.get("ingredients", [])[:8])
    return f"- {md.get('name', 'unknown recipe')} ({md.get('minutes', '?')} min) — ingredients: {ingredients}"

### 4.1 Tool: search_recipes (semantic)

In [ ]:
@tool
def search_recipes(
    query: str,
    top_k: int = 5,
    max_minutes: int | None = None,
    must_have_tag: str | None = None,
    must_have_ingredient: str | None = None,
    max_calories: float | None = None,
    exclude_ingredients: str | None = None,
    dietary_restriction: str | None = None,
) -> str:
    """Semantic search over the recipe knowledge base.

    Use this when the user asks for recipe ideas or dish suggestions that
    benefit from matching descriptions and ingredients.

    You can also pass hard constraints so semantic results do not violate
    requirements: max_minutes, tag, required ingredient, max calories,
    comma-separated excluded ingredients or dietary_restriction.
    """
    must_have_tag, must_have_ingredient, dietary_restriction, exclude_ingredients = (
        infer_recipe_constraints_from_query(
            query=query,
            must_have_tag=must_have_tag,
            must_have_ingredient=must_have_ingredient,
            dietary_restriction=dietary_restriction,
            exclude_ingredients=exclude_ingredients,
        )
    )

    q_vec = embed_model.encode(query, normalize_embeddings=True).tolist()
    fetch_k = max(top_k * 8, 30)
    res = index.query(vector=q_vec, top_k=fetch_k, namespace=RECIPES_NS, include_metadata=True)
    if not res.matches:
        return "No recipes found."

    exclude_terms = expand_dietary_exclusions(exclude_ingredients, dietary_restriction)
    required_ingredient = normalize_terms(must_have_ingredient)
    required_tag = must_have_tag.lower() if must_have_tag else None

    lines = []
    for match in res.matches:
        md = match.metadata
        ingredients = md.get("ingredients", [])
        tags = [str(tag).lower() for tag in md.get("tags", [])]
        minutes = md.get("minutes", None)
        calories = md.get("calories", None)

        if max_minutes is not None and minutes is not None and float(minutes) > max_minutes:
            continue
        if required_tag and required_tag not in tags:
            continue
        if required_ingredient and not contains_any_ingredient(ingredients, required_ingredient):
            continue
        if max_calories is not None and calories is not None and float(calories) > max_calories:
            continue
        if exclude_terms and contains_any_ingredient(ingredients, exclude_terms):
            continue

        lines.append(recipe_metadata_line(md))
        if len(lines) >= top_k:
            break

    if not lines:
        return "No recipes found after applying the requested constraints."
    return "\n".join(lines)

### 4.1 Tool: filter_recipes (structured)

In [ ]:
@tool
def filter_recipes(
    max_minutes: int | None = None,
    must_have_tag: str | None = None,
    must_have_ingredient: str | None = None,
    max_calories: float | None = None,
    exclude_ingredients: str | None = None,
    dietary_restriction: str | None = None,
    limit: int = 10,
) -> str:
    """Structured filter over recipes using exact or numeric constraints.

    Use this when the user has hard requirements: max cooking time, a required
    tag, required ingredient, calorie ceiling, comma-separated excluded ingredients or dietary
    restrictions.

    If memory says the user is lactose intolerant, pass
    dietary_restriction="lactose_intolerant" so dairy ingredients are excluded.
    If the user asks for a dish type or ingredient (for example pasta, chicken,
    chocolate, tomato), pass it as must_have_ingredient.
    """
    df = RECIPES_DF
    mask = pd.Series([True] * len(df), index=df.index)
    if max_minutes is not None:
        mask &= df["minutes"] <= max_minutes
    if must_have_tag is not None:
        tag = must_have_tag.lower()
        mask &= df["tags"].apply(lambda tags: tag in [str(x).lower() for x in tags])
    if must_have_ingredient is not None:
        required_terms = normalize_terms(must_have_ingredient)
        mask &= df["ingredients"].apply(lambda ingredients: contains_any_ingredient(ingredients, required_terms))
    if max_calories is not None:
        mask &= df["nutrition"].apply(lambda values: len(values) == 7 and values[0] <= max_calories)

    exclude_terms = expand_dietary_exclusions(exclude_ingredients, dietary_restriction)
    if exclude_terms:
        mask &= ~df["ingredients"].apply(lambda ingredients: contains_any_ingredient(ingredients, exclude_terms))

    matches = df[mask].head(limit)
    if matches.empty:
        return "No recipes match those filters."

    lines = []
    for _, row in matches.iterrows():
        md = {
            "name": row["name"],
            "minutes": row["minutes"],
            "ingredients": row["ingredients"],
        }
        lines.append(
            f"{recipe_metadata_line(md)} "
            f"({row['n_ingredients']} ingredients, {row['n_steps']} steps)"
        )
    return "\n".join(lines)

### 4.3 Tool: search_papers (semantic)

In [ ]:
@tool
def search_papers(query: str, top_k: int = 6) -> str:
    """Semantic search over the AI papers knowledge base.

    Use this for questions about AI / ML / LLM topics covered in the course
    papers. The returned source labels must be cited exactly in the answer.
    """
    q_vec = embed_model.encode(query, normalize_embeddings=True).tolist()
    res = index.query(vector=q_vec, top_k=top_k, namespace=PAPERS_NS, include_metadata=True)
    if not res.matches:
        return "No relevant content found in the papers."
    lines = []
    for match in res.matches:
        md = match.metadata
        paper = md.get("paper", "unknown paper")
        page_raw = md.get("page", "?")
        try:
            page = int(float(page_raw))
        except Exception:
            page = page_raw
        chunk_idx = md.get("chunk_idx", "?")
        snippet = str(md.get("text", ""))[:360].replace("\n", " ")
        lines.append(
            f"- Source: {paper}, page {page}, chunk {chunk_idx}. "
            f"Snippet: {snippet}..."
        )
    return "\n".join(lines)

### 4.4 Tool: save_memory (long-term, per user)

In [ ]:
@tool
def save_memory(text: str) -> str:
    """Persist a long-term semantic memory about the current user.

    Use this when the user reveals stable facts or preferences that should
    survive across conversations (name, dietary restrictions, allergies,
    favourite cuisines).

    IMPORTANT: combine ALL facts from the current message into a single
    concise third-person sentence. Never call this tool more than once for
    the same user message.
    """
    user_id = current_user_id.get()
    ns = f"{MEMORY_NS_PREFIX}_{user_id}"
    clean_text = " ".join(text.strip().split())
    if not clean_text:
        return "No memory saved: empty text."

    canonical_text = canonical_memory_text(clean_text)
    vec = embed_model.encode(clean_text, normalize_embeddings=True).tolist()

    # Guardrail: avoid saving the same stable fact multiple times.
    try:
        existing = index.query(vector=vec, top_k=8, namespace=ns, include_metadata=True)
        for match in existing.matches:
            stored_text = str(match.metadata.get("text", ""))
            stored_canonical = canonical_memory_text(stored_text)
            semantic_duplicate = float(getattr(match, "score", 0) or 0) >= 0.94
            if stored_canonical == canonical_text or semantic_duplicate:
                return f"Memory already exists for user '{user_id}'."
    except Exception:
        # If the namespace does not exist yet, continue and create the first memory.
        pass

    mem_id = f"mem-{user_id}-{uuid.uuid4().hex[:8]}"
    index.upsert(
        vectors=[{
            "id": mem_id,
            "values": vec,
            "metadata": {"text": clean_text, "user_id": user_id},
        }],
        namespace=ns,
    )
    return f"Memory saved for user '{user_id}'."

### 4.5 Tool: recall_memory (long-term, per user)

In [ ]:
@tool
def recall_memory(query: str = "user profile and preferences", top_k: int = 3) -> str:
    """Retrieve long-term memories about the current user relevant to the query.

    Call this whenever the user references something personal or asks the
    agent to remember details about them. It only sees memories saved for
    the current user — never another user's data.
    """
    user_id = current_user_id.get()
    ns = f"{MEMORY_NS_PREFIX}_{user_id}"
    query = " ".join((query or "user profile and preferences").strip().split())
    vec = embed_model.encode(query, normalize_embeddings=True).tolist()
    try:
        res = index.query(vector=vec, top_k=top_k, namespace=ns, include_metadata=True)
    except Exception:
        return "No memories yet for this user."
    if not res.matches:
        return "No relevant memories found."

    lines = []
    seen = set()
    for match in res.matches:
        text = str(match.metadata.get("text", "")).strip()
        canonical = canonical_memory_text(text)
        if text and canonical not in seen:
            seen.add(canonical)
            lines.append(f"- {text}")

    if not lines:
        return "No relevant memories found."
    return "\n".join(lines)

### 4.6 Tool: list_user_memories and forget_memory

These tools make long-term memory inspectable and user-controllable. They are useful for demos, privacy, and debugging.

In [ ]:
@tool
def list_user_memories(query: str = "user profile and preferences", top_k: int = 10) -> str:
    """List memories currently known for the active user.

    Use this when the user asks "what do you remember about me?" or when we
    need to inspect memory during evaluation.
    """
    return recall_memory.invoke({"query": query, "top_k": top_k})


@tool
def forget_memory(text: str, min_score: float = 0.96) -> str:
    """Delete a matching memory for the active user.

    Use this when the user asks to forget a specific fact. It deletes exact
    text matches or very high-similarity matches.
    """
    user_id = current_user_id.get()
    ns = f"{MEMORY_NS_PREFIX}_{user_id}"
    clean_text = " ".join(text.strip().split())
    if not clean_text:
        return "No memory deleted: empty text."

    vec = embed_model.encode(clean_text, normalize_embeddings=True).tolist()
    try:
        res = index.query(vector=vec, top_k=5, namespace=ns, include_metadata=True)
    except Exception:
        return "No memories yet for this user."

    canonical_target = canonical_memory_text(clean_text)
    ids_to_delete = []
    for match in res.matches:
        stored_text = str(match.metadata.get("text", ""))
        canonical_stored = canonical_memory_text(stored_text)
        if canonical_stored == canonical_target or float(match.score) >= min_score:
            ids_to_delete.append(match.id)

    if not ids_to_delete:
        return "No matching memory found."

    index.delete(ids=ids_to_delete, namespace=ns)
    return f"Deleted {len(ids_to_delete)} memory item(s) for user '{user_id}'."

### 4.7 Collect tools

Single list consumed by the chatbot node in section 5.

In [ ]:
TOOLS = [
    search_recipes,
    filter_recipes,
    search_papers,
    save_memory,
    recall_memory,
    list_user_memories,
    forget_memory,
]
TOOLS_BY_NAME = {tool.name: tool for tool in TOOLS}
print(f"Defined {len(TOOLS)} tools: {[t.name for t in TOOLS]}")

## 5. Agent state and nodes

### 5.1 State definition

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

### 5.2 LLM with tools bound

`bind_tools` exposes the @tool docstrings as the function schema the model uses to issue tool calls.

In [ ]:
from langchain_openai import ChatOpenAI

LLM_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.2

llm = ChatOpenAI(model=LLM_MODEL, temperature=TEMPERATURE)
llm_with_tools = llm.bind_tools(TOOLS)
print(f"LLM ready: {LLM_MODEL} (temperature={TEMPERATURE})")

### 5.3 System prompt

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant for the TT2 Obligatorio.

You have access to TWO knowledge bases and per-user long-term memory:
1. Recipes from Food.com — use `search_recipes` for semantic ideas and `filter_recipes` for hard constraints.
2. Six foundational AI papers/documents — use `search_papers` and cite the exact source labels returned by the tool.
3. Long-term semantic memory per user — use `save_memory`, `recall_memory`, `list_user_memories` and `forget_memory`.

Behavior rules:
- Stable personal info (name, allergies, diet, preferences) should be saved with `save_memory` in one concise sentence.
- Do not save the same user fact twice. If it already exists, acknowledge it without writing a duplicate.
- At the start of substantive interactions, or when the user references personal context, call `recall_memory` first with a short query.
- If memory or the user says lactose intolerant / dairy-free / no milk, treat dairy as a hard exclusion. Pass `dietary_restriction="lactose_intolerant"` to recipe tools.
- For recipe questions, combine `filter_recipes` for hard constraints with `search_recipes` for semantic matching. Pass hard constraints to both tools when possible. For vegetarian/vegan requests, always pass the tag and avoid meat/fish stock leakage.
- In final recipe answers, only recommend recipes that appeared in tool results. Do not invent recipe names, ingredients or cooking times.
- For AI / ML / LLM questions, use `search_papers` and cite paper name plus page exactly as returned.
- If a search returns nothing relevant, say \"I don't have enough information to answer that\" — do NOT invent facts.
- When refusing due to missing evidence, do not mention model knowledge cutoffs or current-date assumptions.
- Use `list_user_memories` when the user asks what you remember about them.
- Use `forget_memory` when the user asks to delete/forget a memory.
- For pure small talk, reply directly without calling any tool.
- Keep answers concise and well-formatted."""

### 5.4 Chatbot node


In [ ]:
def chatbot_node(state: AgentState) -> dict:
    """Single LLM call: system prompt + conversation history → assistant message."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

### 5.5 Tools node (manual)


In [ ]:
def tools_node(state: AgentState) -> dict:
    """Execute every tool call in the latest assistant message and append the results."""
    last = state["messages"][-1]
    tool_messages = []
    for call in last.tool_calls:
        tool = TOOLS_BY_NAME[call["name"]]
        try:
            result = tool.invoke(call["args"])
        except Exception as exc:
            result = f"Tool error: {exc}"
        tool_messages.append(ToolMessage(
            content=str(result),
            tool_call_id=call["id"],
            name=call["name"],
        ))
    return {"messages": tool_messages}

### 5.6 Conditional edge router


In [ ]:
from langgraph.graph import END

def route_after_chatbot(state: AgentState) -> str:
    """Return 'tools' if the LLM asked for tool execution, otherwise END."""
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return END

## 6. Graph construction

### 6.1 Build and compile the graph

In [ ]:
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import MemorySaver

builder = StateGraph(AgentState)
builder.add_node("chatbot", chatbot_node)
builder.add_node("tools",   tools_node)

builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", route_after_chatbot, {"tools": "tools", END: END})
builder.add_edge("tools", "chatbot")

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)
print("Graph compiled.")

### 6.2 Visualize the graph

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Could not render image ({e}). ASCII fallback:")
    print(graph.get_graph().draw_ascii())

## 7. Memory and session helpers

### 7.1 Send a message

In [ ]:
def thread_config(user_id: str, thread_id: str) -> dict:
    """Build the LangGraph config key used for short-term memory.

    The visible thread_id is still easy to read (for example alice-a1b2c3d4),
    but internally we include user_id as well. This prevents two users from
    accidentally sharing short-term memory if they reuse the same thread_id.
    """
    return {"configurable": {"thread_id": f"{user_id}:{thread_id}"}}


def chat(user_id: str, thread_id: str, user_message: str) -> str:
    """Send a message in a given (user, thread) and return the assistant reply."""
    token = current_user_id.set(user_id)
    try:
        result = graph.invoke(
            {"messages": [HumanMessage(content=user_message)]},
            config=thread_config(user_id, thread_id),
        )
    finally:
        current_user_id.reset(token)
    return result["messages"][-1].content

In [ ]:
def chat_verbose(user_id: str, thread_id: str, user_message: str) -> str:
    """Send a message and print intermediate tool calls before returning the final reply."""
    token = current_user_id.set(user_id)
    config = thread_config(user_id, thread_id)
    try:
        # Determine how many messages there were before
        state_before = graph.get_state(config=config)
        n_before = len(state_before.values.get("messages", [])) if state_before.values else 0

        result = graph.invoke(
            {"messages": [HumanMessage(content=user_message)]},
            config=config,
        )
    finally:
        current_user_id.reset(token)

    # Show only new messages
    new_messages = result["messages"][n_before:]
    for msg in new_messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"Tool: {tc['name']} | args: {tc['args']}")
        elif hasattr(msg, "name") and msg.name:
            print(f"Result of {msg.name}: {str(msg.content)[:700]}...")

    return result["messages"][-1].content

### 7.2 Conversation registry


In [ ]:
_threads_per_user: dict[str, list[str]] = {}

def new_thread(user_id: str) -> str:
    """Create a new conversation id for the user and register it."""
    tid = f"{user_id}-{uuid.uuid4().hex[:8]}"
    _threads_per_user.setdefault(user_id, []).append(tid)
    return tid

def list_threads(user_id: str) -> list[str]:
    """Return all known thread ids for a user (newest last)."""
    return list(_threads_per_user.get(user_id, []))

### 7.3 Inspect a conversation


In [ ]:
def infer_user_from_thread(thread_id: str) -> str:
    """Best-effort helper for old demo calls where only thread_id is passed."""
    return thread_id.split("-", 1)[0] if "-" in thread_id else "default_user"


def print_thread(thread_id: str, user_id: str | None = None) -> None:
    """Print every message currently stored under a user/thread pair."""
    user_id = user_id or infer_user_from_thread(thread_id)
    snap = graph.get_state(config=thread_config(user_id, thread_id))
    if not snap or not snap.values.get("messages"):
        print(f"(thread {thread_id} is empty for user {user_id})")
        return
    for m in snap.values["messages"]:
        role = type(m).__name__.replace("Message", "")
        content = (m.content or "").replace("\n", " ")[:200]
        print(f"  [{role}] {content}")

### 8. Demo

### 8.1 Dataset Q&A - semantic + structured

In [ ]:
alice_t1 = new_thread("alice")
print(chat_verbose("alice", alice_t1, "Give me 3 quick vegetarian pasta dishes, under 30 minutes."))

### 8.2 Papers Q&A

In [ ]:
print(chat_verbose("alice", alice_t1, "Summarize what multi-head attention is, citing the paper."))

### 8.3 Small talk (no tools)

In [ ]:
print(chat_verbose("alice", alice_t1, "Hi, how are you today?"))

### 8.4 Save a long-term memory

In [ ]:
print(chat_verbose("alice", alice_t1, "By the way, please remember I'm lactose intolerant and I love spicy food."))

### 8.5 New conversation, same user — memory persists

In [ ]:
alice_t2 = new_thread("alice")
print(chat_verbose("alice", alice_t2, "Suggest a spicy dinner I would enjoy under 60 minutes, based on what you remember about me."))

### 8.6 Different user — no leakage from Alice's memory

In [ ]:
bob_t1 = new_thread("bob")
print(chat_verbose("bob", bob_t1, "What do you remember about me?"))

### 8.7 Refusal when the agent does not know

In [ ]:
print(chat_verbose("bob", bob_t1, "Who won the 2030 FIFA World Cup?"))

### 8.8 Resume a paused conversation

In [ ]:
print("Threads for alice:", list_threads("alice"))
print("\nResuming Alice's first thread:")
print(chat_verbose("alice", alice_t1, "What did I ask you first in this conversation?"))
print("\nFull message log of alice_t1:")
print_thread(alice_t1)

## 9. Gradio UI

In [ ]:
import gradio as gr

def gradio_respond(message, chat_history, user_id, thread_id):
    """Glue between the Gradio chat widget and our `chat` function."""
    user_id = (user_id or "anon").strip() or "anon"
    if not thread_id.strip():
        thread_id = new_thread(user_id)

    reply = chat(user_id, thread_id, message)
    chat_history = (chat_history or []) + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply},
    ]
    return "", chat_history, thread_id

with gr.Blocks(title="Agentic RAG — Recipes + Papers") as demo:
    gr.Markdown("# Agentic RAG — Recipes + Papers")
    gr.Markdown("Set a `User ID` to scope long-term memory. Leave `Thread ID` empty to start a new conversation.")
    with gr.Row():
        user_box   = gr.Textbox(label="User ID",   value="alice", scale=1)
        thread_box = gr.Textbox(label="Thread ID", value="",       scale=2)
    chat_widget = gr.Chatbot(height=420, type="messages", allow_tags=False)
    msg = gr.Textbox(label="Your message", placeholder="Ask about recipes, papers, or anything…")
    clear = gr.Button("New conversation")

    msg.submit(gradio_respond, [msg, chat_widget, user_box, thread_box], [msg, chat_widget, thread_box])
    clear.click(lambda: ([], ""), None, [chat_widget, thread_box])

demo.launch(share=True)

## 10. Evaluation and requirement checks

This section is intentionally small and practical. It gives us a repeatable way to show that the system covers the assignment requirements:

- dataset questions;
- paper/document questions;
- long-term memory;
- user isolation;
- short-term conversation context;
- refusal when there is not enough information.

Run these cells after the graph and tools have been built.

In [ ]:
EVAL_CASES = [
    {
        "name": "Save long-term memory",
        "user_id": "eval_alice",
        "message": "Remember that I am lactose intolerant and I like spicy food.",
        "checks": ["calls save_memory once or reports duplicate", "stores stable user preference"],
    },
    {
        "name": "Recall memory and apply dietary restriction",
        "user_id": "eval_alice",
        "message": "Suggest a spicy dinner under 60 minutes based on what you remember about me.",
        "checks": ["calls recall_memory", "uses lactose memory", "excludes dairy ingredients", "uses recipe tools"],
    },
    {
        "name": "Inspect user memory",
        "user_id": "eval_alice",
        "message": "What do you remember about me?",
        "checks": ["uses list_user_memories or recall_memory", "shows only Alice memories"],
    },
    {
        "name": "User isolation",
        "user_id": "eval_bob",
        "message": "What do you remember about me?",
        "checks": ["does not reveal Alice memories"],
    },
    {
        "name": "Recipe query with hard constraints",
        "user_id": "eval_bob",
        "message": "Give me 2 vegetarian pasta ideas under 30 minutes.",
        "checks": ["uses recipe tools", "respects time/tag/ingredient constraints", "does not invent recipes"],
    },
    {
        "name": "Paper-grounded answer",
        "user_id": "eval_bob",
        "message": "Explain multi-head attention and cite the paper page.",
        "checks": ["uses search_papers", "cites paper/page", "does not answer from memory only"],
    },
    {
        "name": "Unknown future event",
        "user_id": "eval_bob",
        "message": "Who won the 2030 FIFA World Cup?",
        "checks": ["admits lack of information", "does not invent"],
    },
]

pd.DataFrame(EVAL_CASES)

In [ ]:
def run_eval_cases(cases: list[dict]) -> None:
    """Run the evaluation cases and print tool traces plus final answers.

    This is not an automatic grader. It is a demo/evaluation harness: each case
    prints the expected checks so we can manually verify behavior during the
    defense or while iterating on the notebook.
    """
    threads_by_user = {}
    for case in cases:
        user_id = case["user_id"]
        thread_id = threads_by_user.get(user_id)
        if thread_id is None or case["name"] == "Recall memory and apply dietary restriction":
            thread_id = new_thread(user_id)
            threads_by_user[user_id] = thread_id

        print("=" * 90)
        print(f"CASE: {case['name']}")
        print(f"USER: {user_id} | THREAD: {thread_id}")
        print(f"MESSAGE: {case['message']}")
        print(f"EXPECTED CHECKS: {', '.join(case['checks'])}")
        print("-" * 90)
        answer = chat_verbose(user_id, thread_id, case["message"])
        print("\nFINAL ANSWER:")
        print(answer)


# Set to False if you want to avoid extra OpenAI calls while editing.
RUN_FULL_EVAL = True
if RUN_FULL_EVAL:
    run_eval_cases(EVAL_CASES)

### 10.1 What to report from the evaluation

For the written documentation or defense, record:

1. Which tools were called in each case.
2. Whether the answer used retrieved evidence.
3. Whether user memory was isolated correctly.
4. Any failure cases, especially hallucinated recipes or irrelevant memories.
5. What we changed after observing failures.

This directly supports the required documentation section: "Pruebas y resultados obtenidos para chatbot".